# Smart Beta 完整教程：从概念到实战

## 📚 教学目标
1. 理解 Smart Beta 的定义：介于纯被动和纯主动之间的因子投资
2. 掌握常见 Smart Beta 策略：等权、最小方差、风险平价、基本面加权、高股息
3. 实现各策略的权重计算
4. 对比不同策略的因子暴露、收益和风险特征
5. 理解 A 股 Smart Beta ETF 产品生态

**参考书**：《因子投资：方法与实践》（石川）第 7.4 节
**教学策略**：先用极小数据集让你"看见"每一步计算，再扩展到真实规模

---

## 1. 什么是 Smart Beta？

### 🎯 一个直觉问题

假设你知道"小市值股票收益更高"和"高 B/M 股票收益更高"这两个学术发现。

作为普通投资者，你会怎么做？

- **被动投资**：买入市值加权的宽基指数 → 无法利用这些发现
- **主动投资**：自己选股 → 需要专业知识、高费用、高换手率
- **Smart Beta**：买入专门暴露于这些因子的指数基金 → 简单、低成本、规则透明

### 📖 书中的定义

> Smart Beta 是在某类资产内，以更高的收益或更低的风险为目标，针对一个或多个具有风险溢价的因子来确定投资组合持仓及权重。

### 📐 Smart Beta 的四个要素

| 要素 | 说明 |
|------|------|
| 偏离市值加权 | 不同于传统被动指数的市值加权 |
| 规则透明 | 编制方案清晰可复制 |
| 因子暴露 | 主动在一个或多个因子上暴露 |
| 增强收益/降低风险 | 追求比基准更好的投资结果 |

### 🔗 被动投资、主动投资与 Smart Beta 的关系

```
被动投资 ←───────────── Smart Beta ─────────────→ 主动投资
(市值加权)        (另类加权/因子筛选)          (主观选股)
低费用                                        高费用
规则透明                                      主观判断
低换手率                                      高换手率
仅捕获市场溢价                               捕获多种因子溢价
```

接下来，我们从最小的例子开始，一步步实现各种 Smart Beta 策略。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize

plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

np.random.seed(42)
print("✅ 库导入完成")

---

## 2. 从 10 只股票开始：手算各策略权重

### 🎯 场景

假设整个市场只有 **10 只股票**。我们有每只股票的：
- **市值**（亿元）
- **账面市值比 B/M**
- **股息率**
- **过去 12 个月收益率**（用于计算波动率）

我们来比较 5 种常见的 Smart Beta 策略如何分配权重。

In [ ]:
# ========== 构造 10 只股票的微型数据集 ==========
mini_data = pd.DataFrame({
    'Stock': ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J'],
    'Market_Cap': [5, 12, 18, 25, 35, 48, 60, 78, 90, 120],
    'B2M': [2.1, 1.8, 1.5, 1.2, 1.0, 0.8, 0.6, 0.5, 0.4, 0.3],
    'Dividend_Yield': [0.06, 0.05, 0.04, 0.03, 0.02, 0.03, 0.04, 0.05, 0.03, 0.02],
    'ROE': [0.15, 0.12, 0.18, 0.10, 0.08, 0.20, 0.16, 0.14, 0.11, 0.09]
})

# 模拟过去 12 个月的月收益率（用于计算波动率）
np.random.seed(42)
monthly_rets = pd.DataFrame({
    stock: np.random.normal(ret, 5, 12)
    for stock, ret in zip(
        mini_data['Stock'],
        [8.2, 6.5, 5.1, 4.8, 3.2, 2.5, 1.8, 0.5, -0.3, -1.2]
    )
})

# 计算年化波动率
volatility = monthly_rets.std() * np.sqrt(12)
mini_data['Volatility'] = mini_data['Stock'].map(volatility).values

print("📋 原始数据：")
print(mini_data.to_string(index=False))
print(f"\n💡 数据特征：")
print(f"   市值范围: {mini_data['Market_Cap'].min()} ~ {mini_data['Market_Cap'].max()} 亿")
print(f"   B/M 范围: {mini_data['B2M'].min():.1f} ~ {mini_data['B2M'].max():.1f}")
print(f"   波动率范围: {mini_data['Volatility'].min():.1f}% ~ {mini_data['Volatility'].max():.1f}%")

### 📐 策略 1：等权重 (Equal Weight)

最简单的 Smart Beta 策略：每只股票权重相同。

$$w_i = \frac{1}{N}$$

其中：
- $N$ = 股票总数
- $w_i$ = 股票 $i$ 的权重

**特点**：
- ✅ 最简单、最透明
- ✅ 隐含偏向小市值股票（小市值股在组合中的实际权重高于市值加权）
- ⚠️ 换手率较高（每期需重新平衡）

In [ ]:
# ========== 策略 1: 等权重 ==========
N = len(mini_data)
mini_data['EW_Weight'] = 1.0 / N

print("📊 步骤 1: 等权重策略")
print("─" * 50)
print(f"  每只股票权重 = 1 / {N} = {1/N:.4f} = {1/N*100:.1f}%")
print()
print("  结果:")
for _, row in mini_data.iterrows():
    print(f"    {row['Stock']}: w = {row['EW_Weight']:.4f} ({row['EW_Weight']*100:.1f}%)")
print(f"\n  ✅ 权重之和 = {mini_data['EW_Weight'].sum():.4f}")
print(f"\n  💡 等权重策略隐含偏向小市值股票：")
print(f"     小市值股 A 在市值加权中权重 = {5/sum(mini_data['Market_Cap'])*100:.1f}%")
print(f"     小市值股 A 在等权重中权重   = {1/N*100:.1f}%")
print(f"     → 等权重策略实际上在规模因子上有正向暴露！")

### 📐 策略 2：最小方差 (Minimum Variance)

目标：找到使组合波动率最小的权重。

$$\min_w \quad w' \Sigma w$$
$$\text{s.t.} \quad \sum w_i = 1, \quad w_i \geq 0$$

其中：
- $w$ = 权重向量
- $\Sigma$ = 股票收益率的协方差矩阵

**特点**：
- ✅ 直接降低组合风险
- ✅ 隐含偏向低波动率股票
- ⚠️ 需要估计协方差矩阵（估计误差风险）

In [ ]:
# ========== 策略 2: 最小方差 ==========
# 计算协方差矩阵
cov_matrix = monthly_rets.cov()

print("📊 步骤 2: 最小方差策略")
print("─" * 50)
print(f"  目标: min w'Σw, s.t. Σw=1, w≥0")
print()

# 使用 scipy.optimize 求解
def portfolio_variance(w, cov):
    return w @ cov.values @ w

constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
bounds = [(0, 1) for _ in range(N)]
w0 = np.ones(N) / N  # 初始值：等权重

result = minimize(portfolio_variance, w0, args=(cov_matrix,),
                  method='SLSQP', bounds=bounds, constraints=constraints)

mini_data['MV_Weight'] = result.x

print("  结果:")
for _, row in mini_data.iterrows():
    bar = '█' * int(row['MV_Weight'] * 50)
    print(f"    {row['Stock']}: w = {row['MV_Weight']:.4f} ({row['MV_Weight']*100:.1f}%)  {bar}")
print(f"\n  ✅ 权重之和 = {mini_data['MV_Weight'].sum():.4f}")
print(f"  ✅ 组合波动率 = {np.sqrt(result.fun * 12):.2f}% (年化)")

# 与等权重对比
ew_var = np.ones(N)/N @ cov_matrix.values @ np.ones(N)/N
print(f"\n  💡 对比：等权重组合波动率 = {np.sqrt(ew_var * 12):.2f}% (年化)")
print(f"     最小方差策略降低了 {np.sqrt(ew_var * 12) - np.sqrt(result.fun * 12):.2f}% 的波动率")

### 📐 策略 3：风险平价 (Risk Parity)

目标：使每只股票对组合风险的贡献相等。

$$w_i \times \frac{\partial \sigma_p}{\partial w_i} = \frac{\sigma_p}{N}$$

简化形式（忽略相关性时）：

$$w_i = \frac{1/\sigma_i}{\sum_{j=1}^N 1/\sigma_j}$$

其中：
- $\sigma_i$ = 股票 $i$ 的波动率
- $\sigma_p$ = 组合波动率

**特点**：
- ✅ 风险分散更均匀
- ✅ 不需要估计预期收益率
- ⚠️ 低波动率股票获得高权重

In [ ]:
# ========== 策略 3: 风险平价（简化版） ==========
print("📊 步骤 3: 风险平价策略")
print("─" * 50)
print(f"  公式: w_i = (1/σ_i) / Σ(1/σ_j)")
print()

# 计算每只股票的波动率
sigma = mini_data['Volatility'].values
inv_sigma = 1.0 / sigma
mini_data['RP_Weight'] = inv_sigma / inv_sigma.sum()

print("  详细计算:")
for _, row in mini_data.iterrows():
    print(f"    {row['Stock']}: σ={row['Volatility']:.2f}%, 1/σ={1/row['Volatility']:.4f}, w={row['RP_Weight']:.4f}")

print(f"\n  结果:")
for _, row in mini_data.iterrows():
    bar = '█' * int(row['RP_Weight'] * 50)
    print(f"    {row['Stock']}: w = {row['RP_Weight']:.4f} ({row['RP_Weight']*100:.1f}%)  {bar}")
print(f"\n  ✅ 权重之和 = {mini_data['RP_Weight'].sum():.4f}")

# 验证风险贡献
port_vol = np.sqrt(mini_data['RP_Weight'].values @ cov_matrix.values @ mini_data['RP_Weight'].values * 12)
marginal_risk = cov_matrix.values @ mini_data['RP_Weight'].values * 12 / port_vol
risk_contrib = mini_data['RP_Weight'].values * marginal_risk
print(f"\n  💡 各股票的风险贡献（理想情况下应相等）:")
for stock, rc in zip(mini_data['Stock'], risk_contrib):
    print(f"    {stock}: 风险贡献 = {rc:.4f} ({rc/port_vol*100:.1f}%)")

### 📐 策略 4：基本面加权 (Fundamental Weighting)

按基本面指标（如 B/M、ROE、股息率等）确定权重。

以 B/M 加权为例：

$$w_i = \frac{B/M_i}{\sum_{j=1}^N B/M_j}$$

**特点**：
- ✅ 隐含偏向价值股（高 B/M）
- ✅ 规则简单透明
- ⚠️ 因子选择影响结果

In [ ]:
# ========== 策略 4: 基本面加权 (B/M) ==========
print("📊 步骤 4: 基本面加权策略 (B/M)")
print("─" * 50)
print(f"  公式: w_i = B/M_i / Σ(B/M_j)")
print()

bm = mini_data['B2M'].values
mini_data['FW_Weight'] = bm / bm.sum()

print("  详细计算:")
for _, row in mini_data.iterrows():
    print(f"    {row['Stock']}: B/M={row['B2M']:.2f}, w={row['B2M']:.2f}/{bm.sum():.2f}={row['FW_Weight']:.4f}")

print(f"\n  结果:")
for _, row in mini_data.iterrows():
    bar = '█' * int(row['FW_Weight'] * 50)
    print(f"    {row['Stock']}: w = {row['FW_Weight']:.4f} ({row['FW_Weight']*100:.1f}%)  {bar}")
print(f"\n  ✅ 权重之和 = {mini_data['FW_Weight'].sum():.4f}")
print(f"\n  💡 高 B/M 股票（如 A、B）获得更高权重 → 价值因子暴露")

### 📐 策略 5：高股息加权 (Dividend Weighting)

按股息率确定权重：

$$w_i = \frac{DY_i}{\sum_{j=1}^N DY_j}$$

其中：
- $DY_i$ = 股票 $i$ 的股息率

**特点**：
- ✅ 偏向高分红股票（防御性）
- ✅ 适合追求稳定现金流的投资者
- ⚠️ 行业集中度可能较高（如银行、公用事业）

In [ ]:
# ========== 策略 5: 高股息加权 ==========
print("📊 步骤 5: 高股息加权策略")
print("─" * 50)
print(f"  公式: w_i = DY_i / Σ(DY_j)")
print()

dy = mini_data['Dividend_Yield'].values
mini_data['DY_Weight'] = dy / dy.sum()

print("  详细计算:")
for _, row in mini_data.iterrows():
    print(f"    {row['Stock']}: DY={row['Dividend_Yield']:.2%}, w={row['Dividend_Yield']:.4f}/{dy.sum():.4f}={row['DY_Weight']:.4f}")

print(f"\n  结果:")
for _, row in mini_data.iterrows():
    bar = '█' * int(row['DY_Weight'] * 50)
    print(f"    {row['Stock']}: w = {row['DY_Weight']:.4f} ({row['DY_Weight']*100:.1f}%)  {bar}")
print(f"\n  ✅ 权重之和 = {mini_data['DY_Weight'].sum():.4f}")
print(f"\n  💡 高股息率股票（如 A: 6%, B: 5%）获得更高权重 → 红利因子暴露")

---

## 3. 可视化：对比五种策略的权重分配

In [ ]:
# ========== 可视化：五种策略权重对比 ==========
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

strategies = [
    ('EW_Weight', 'Equal Weight', '#3498db'),
    ('MV_Weight', 'Minimum Variance', '#2ecc71'),
    ('RP_Weight', 'Risk Parity', '#e67e22'),
    ('FW_Weight', 'Fundamental (B/M)', '#9b59b6'),
    ('DY_Weight', 'Dividend Weight', '#e74c3c'),
]

stocks = mini_data['Stock'].values
x = np.arange(len(stocks))

for idx, (col, title, color) in enumerate(strategies):
    ax = axes[idx // 3][idx % 3]
    bars = ax.bar(x, mini_data[col].values, color=color, edgecolor='black', alpha=0.85)
    ax.set_xlabel('Stock', fontsize=11)
    ax.set_ylabel('Weight', fontsize=11)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(stocks)
    ax.grid(axis='y', alpha=0.3)

    # 标注数值
    for bar, v in zip(bars, mini_data[col].values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                f'{v:.3f}', ha='center', va='bottom', fontsize=8)

# 对比柱状图
ax = axes[1][2]
width = 0.15
for i, (col, title, color) in enumerate(strategies):
    ax.bar(x + i * width, mini_data[col].values, width, label=title, color=color, alpha=0.85)
ax.set_xlabel('Stock', fontsize=11)
ax.set_ylabel('Weight', fontsize=11)
ax.set_title('All Strategies Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x + width * 2)
ax.set_xticklabels(stocks)
ax.legend(fontsize=8, loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 观察：")
print("   1. 等权重：所有股票权重相同（10%）")
print("   2. 最小方差：低波动率股票获得更高权重")
print("   3. 风险平价：与最小方差类似，但更均匀")
print("   4. 基本面加权：高 B/M 股票（A、B）权重最高")
print("   5. 高股息：高股息率股票（A、B）权重最高")

---

## 4. 扩展到完整规模：多期模拟

现在我们模拟 **200 只股票 × 60 个月**的数据，比较各策略的长期表现。

### 📐 数据生成过程 (DGP)

- **市值**：对数正态分布
- **B/M**：与市值负相关
- **波动率**：与市值负相关（小市值股票波动更高）
- **收益率**：包含市值效应（负向）、B/M 效应（正向）和噪声

In [ ]:
# ========== 多期模拟 ==========
np.random.seed(42)
N_STOCKS = 200
N_MONTHS = 60

# 存储各策略的月度收益
strategy_returns = {
    'Market_Cap_Weighted': [],
    'Equal_Weight': [],
    'Minimum_Variance': [],
    'Risk_Parity': [],
    'Fundamental_BM': [],
    'Dividend_Weight': [],
}

print(f"📊 模拟参数：")
print(f"  股票数量: {N_STOCKS} 只/月")
print(f"  时间跨度: {N_MONTHS} 个月")
print(f"  策略数量: {len(strategy_returns)} 种")
print(f"\n🔄 开始逐月计算...")

for t in range(N_MONTHS):
    # --- 生成截面数据 ---
    market_cap = np.random.lognormal(mean=10, sigma=1, size=N_STOCKS)
    log_cap = np.log(market_cap)

    # B/M 与市值负相关
    btm = 1.5 - 0.08 * (log_cap - log_cap.mean()) + np.random.normal(0, 0.5, N_STOCKS)
    btm = np.clip(btm, 0.1, 3.0)

    # 股息率（与市值正相关，大公司分红多）
    div_yield = 0.02 + 0.01 * (log_cap - log_cap.mean()) / log_cap.std() + np.random.normal(0, 0.01, N_STOCKS)
    div_yield = np.clip(div_yield, 0.005, 0.08)

    # 波动率（与市值负相关）
    vol = 30 - 3 * (log_cap - log_cap.mean()) / log_cap.std() + np.random.normal(0, 5, N_STOCKS)
    vol = np.clip(vol, 10, 60)

    # 收益率：市值效应(负) + B/M效应(正) + 噪声
    ret = (1.0
           - 0.2 * (log_cap - log_cap.mean())
           + 1.5 * (btm - btm.mean())
           + np.random.normal(0, 5, N_STOCKS))

    # --- 市值加权 ---
    w_mcap = market_cap / market_cap.sum()
    strategy_returns['Market_Cap_Weighted'].append(w_mcap @ ret)

    # --- 等权重 ---
    w_ew = np.ones(N_STOCKS) / N_STOCKS
    strategy_returns['Equal_Weight'].append(w_ew @ ret)

    # --- 最小方差（简化：用历史波动率的倒数） ---
    inv_vol = 1.0 / vol
    w_mv = inv_vol / inv_vol.sum()
    strategy_returns['Minimum_Variance'].append(w_mv @ ret)

    # --- 风险平价 ---
    w_rp = inv_vol / inv_vol.sum()  # 简化版同最小方差
    strategy_returns['Risk_Parity'].append(w_rp @ ret)

    # --- 基本面加权 (B/M) ---
    w_fw = btm / btm.sum()
    strategy_returns['Fundamental_BM'].append(w_fw @ ret)

    # --- 高股息加权 ---
    w_dy = div_yield / div_yield.sum()
    strategy_returns['Dividend_Weight'].append(w_dy @ ret)

    if t < 3:
        print(f"  月份 {t+1}: 各策略收益 = {', '.join([f'{k[:3]}:{v[-1]:.2f}%' for k, v in strategy_returns.items()])}")

print(f"\n  ... (省略第 4~{N_MONTHS} 月) ...")
print(f"\n✅ 完成！共 {N_MONTHS} 个月的策略收益数据")

---

## 5. 各策略的风险收益特征对比

In [ ]:
# ========== 计算各策略的风险收益指标 ==========
print("📊 各策略的风险收益特征")
print("═" * 80)

results = []
for name, rets in strategy_returns.items():
    rets_arr = np.array(rets)
    ann_ret = np.mean(rets_arr) * 12
    ann_vol = np.std(rets_arr, ddof=1) * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0

    # 最大回撤
    cum_ret = np.cumprod(1 + rets_arr / 100)
    peak = np.maximum.accumulate(cum_ret)
    drawdown = (cum_ret - peak) / peak
    max_dd = drawdown.min() * 100

    results.append({
        'Strategy': name,
        'Ann Return (%)': ann_ret,
        'Ann Volatility (%)': ann_vol,
        'Sharpe Ratio': sharpe,
        'Max Drawdown (%)': max_dd
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# 找出最佳策略
best_sharpe = results_df.loc[results_df['Sharpe Ratio'].idxmax()]
print(f"\n🏆 最高 Sharpe Ratio: {best_sharpe['Strategy']} ({best_sharpe['Sharpe Ratio']:.3f})")

In [ ]:
# ========== 可视化：累计收益率曲线 ==========
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- 左图：累计收益率 ---
ax1 = axes[0]
colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6', '#e74c3c', '#1abc9c']
for (name, rets), color in zip(strategy_returns.items(), colors):
    cum_ret = np.cumprod(1 + np.array(rets) / 100)
    ax1.plot(range(1, N_MONTHS + 1), cum_ret, label=name, linewidth=2, color=color)

ax1.set_xlabel('Month', fontsize=12)
ax1.set_ylabel('Cumulative Return (Base=1)', fontsize=12)
ax1.set_title('Smart Beta Strategies: Cumulative Returns', fontsize=14, fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# --- 右图：风险收益散点图 ---
ax2 = axes[1]
for idx, row in results_df.iterrows():
    ax2.scatter(row['Ann Volatility (%)'], row['Ann Return (%)'],
                s=150, color=colors[idx], edgecolors='black', zorder=5)
    ax2.annotate(row['Strategy'], (row['Ann Volatility (%)'], row['Ann Return (%)']),
                textcoords="offset points", xytext=(5, 5), fontsize=9)

ax2.set_xlabel('Annualized Volatility (%)', fontsize=12)
ax2.set_ylabel('Annualized Return (%)', fontsize=12)
ax2.set_title('Risk-Return Scatter', fontsize=14, fontweight='bold')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 观察：")
print("   1. 市值加权策略：基准，捕获市场风险溢价")
print("   2. 等权重策略：偏向小市值，收益可能更高但波动也更大")
print("   3. 最小方差/风险平价：波动率最低，适合风险厌恶型投资者")
print("   4. 基本面加权：偏向价值股，长期收益可能更高")
print("   5. 高股息：防御性较强，适合追求稳定现金流的投资者")

---

## 6. 因子暴露分析

Smart Beta 的核心是在特定因子上获得暴露。我们来分析各策略的因子暴露。

### 📐 因子暴露计算

使用 Fama-MacBeth 回归：

$$r_{i,t} = \alpha_i + \beta_{i,1} f_{1,t} + \beta_{i,2} f_{2,t} + \epsilon_{i,t}$$

其中 $f_{1,t}$ 为市值因子，$f_{2,t}$ 为价值因子。

In [ ]:
# ========== 计算各策略的因子暴露 ==========
# 模拟因子收益率
np.random.seed(42)
factor_mkt = np.random.normal(0.8, 5, N_MONTHS)  # 市场因子
factor_smb = np.random.normal(0.3, 3, N_MONTHS)   # 市值因子 (Small Minus Big)
factor_hml = np.random.normal(0.4, 3, N_MONTHS)   # 价值因子 (High Minus Low)

print("📊 各策略的因子暴露（回归系数）")
print("═" * 70)

factor_exposures = {}
for name, rets in strategy_returns.items():
    rets_arr = np.array(rets)

    # 多因子回归
    X = np.column_stack([np.ones(N_MONTHS), factor_mkt, factor_smb, factor_hml])
    beta = np.linalg.lstsq(X, rets_arr, rcond=None)[0]

    factor_exposures[name] = {
        'Alpha': beta[0],
        'Market': beta[1],
        'SMB': beta[2],
        'HML': beta[3]
    }

exposure_df = pd.DataFrame(factor_exposures).T
print(exposure_df.round(4))

print(f"\n💡 解读：")
print(f"   Market (β): 接近 1 表示与市场同步")
print(f"   SMB (β): 正值表示偏向小市值")
print(f"   HML (β): 正值表示偏向价值股（高 B/M）")

In [ ]:
# ========== 可视化：因子暴露热力图 ==========
fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(exposure_df, annot=True, fmt='.3f', cmap='RdYlGn',
            linewidths=1, ax=ax, center=0,
            cbar_kws={'label': 'Factor Exposure (Beta)'})

ax.set_title('Smart Beta Strategies: Factor Exposures', fontsize=14, fontweight='bold')
ax.set_ylabel('Strategy', fontsize=12)
ax.set_xlabel('Factor', fontsize=12)

plt.tight_layout()
plt.show()

print("\n💡 观察：")
print("   1. 所有策略的 Market Beta 接近 1 → 都是股票组合")
print("   2. 等权重策略：SMB β > 0 → 偏向小市值")
print("   3. 基本面加权：HML β > 0 → 偏向价值股")
print("   4. 最小方差/风险平价：SMB β < 0 → 偏向大市值（大公司波动低）")

---

## 7. 明晟质量指数案例：Smart Beta 指数编制流程

### 📖 书中的案例

> 明晟质量指数（MSCI Quality Index）的编制流程：
> 1. 确定股票范围（NASDAQ、NYSE、AMEX）
> 2. 计算因子得分（ROE、D/E、盈利稳定性）
> 3. 选择得分最高的 N 支股票
> 4. 按质量得分和市值确定权重

### 📐 质量得分计算

$$Z_{i,j} = \frac{x_{i,j} - \bar{x}_j}{\sigma_j}$$

$$Q_i = \frac{Z_{i,ROE} - Z_{i,D/E} - Z_{i,Stability}}{3}$$

### 📐 权重计算（市值调整法）

$$w_i = \frac{s_i \times Q_i}{\sum_{j \in I} s_j \times Q_j}$$

其中：
- $s_i$ = 股票 $i$ 的市值
- $Q_i$ = 质量得分
- $I$ = 选出的 N 支股票集合

In [ ]:
# ========== 模拟明晟质量指数编制 ==========
print("📊 模拟明晟质量指数编制流程")
print("═" * 70)

# 使用 10 只股票数据
df_quality = mini_data[['Stock', 'Market_Cap', 'ROE']].copy()

# 模拟 D/E 和盈利稳定性
np.random.seed(42)
df_quality['DE'] = np.random.uniform(0.2, 1.5, N)
df_quality['Stability'] = np.random.uniform(0.05, 0.3, N)

# 步骤 1: 标准化 (Z-Score)
print("\n📊 步骤 1: 计算 Z-Score")
print("─" * 50)
for col in ['ROE', 'DE', 'Stability']:
    df_quality[f'Z_{col}'] = (df_quality[col] - df_quality[col].mean()) / df_quality[col].std()
    print(f"  Z_{col}: mean={df_quality[f'Z_{col}'].mean():.4f}, std={df_quality[f'Z_{col}'].std():.4f}")

# 步骤 2: 计算质量得分
print("\n📊 步骤 2: 计算质量得分 Q")
print("─" * 50)
print(f"  Q = (Z_ROE - Z_DE - Z_Stability) / 3")
df_quality['Q_Score'] = (df_quality['Z_ROE'] - df_quality['Z_DE'] - df_quality['Z_Stability']) / 3

print("\n  结果:")
for _, row in df_quality.iterrows():
    print(f"    {row['Stock']}: Q = ({row['Z_ROE']:.3f} - {row['Z_DE']:.3f} - {row['Z_Stability']:.3f}) / 3 = {row['Q_Score']:.4f}")

# 步骤 3: 按质量得分排序，选择前 N 只
N_SELECT = 5
df_quality = df_quality.sort_values('Q_Score', ascending=False)
selected = df_quality.head(N_SELECT)

print(f"\n📊 步骤 3: 选择得分最高的 {N_SELECT} 只股票")
print("─" * 50)
for _, row in selected.iterrows():
    print(f"    {row['Stock']}: Q = {row['Q_Score']:.4f}")

# 步骤 4: 按市值调整法确定权重
print(f"\n📊 步骤 4: 市值调整法确定权重")
print("─" * 50)
print(f"  公式: w_i = (s_i × Q_i) / Σ(s_j × Q_j)")

selected = selected.copy()
selected['SQ'] = selected['Market_Cap'] * selected['Q_Score']
selected['Weight'] = selected['SQ'] / selected['SQ'].sum()

print("\n  结果:")
for _, row in selected.iterrows():
    bar = '█' * int(row['Weight'] * 50)
    print(f"    {row['Stock']}: s={row['Market_Cap']:.0f}亿, Q={row['Q_Score']:.3f}, w={row['Weight']:.4f}  {bar}")

print(f"\n  ✅ 权重之和 = {selected['Weight'].sum():.4f}")
print(f"\n  💡 市值调整法同时考虑了：")
print(f"     1. 质量得分（因子暴露）")
print(f"     2. 市值大小（可投资性/流动性）")
print(f"     → 在因子暴露和可投资性之间取得平衡")

---

## 8. 混合法 vs 整合法

### 📖 书中的讨论

> 将多个单一因子进行结合有两种方法：
> - **混合法 (Mix)**：每个因子独立选股，持有多个单因子基金
> - **整合法 (Integrate)**：综合考虑多因子表现，选择一支多因子基金

### 📐 两种方法对比

| 特性 | 混合法 | 整合法 |
|------|--------|--------|
| 操作 | 持有多个单因子 Smart Beta | 持有一个多因子 Smart Beta |
| 因子暴露 | 每个组合独立暴露 | 综合暴露 |
| 持仓数量 | 多（不同因子组合重叠少） | 少 |
| 权重分配 | 需要自己决定因子权重 | 基金内部处理 |
| 优点 | 简单直观、可灵活调整 | 一次性解决 |
| 缺点 | 需要管理多个组合 | 无法灵活调整因子暴露 |

In [ ]:
# ========== 混合法 vs 整合法 ==========
print("📊 混合法 vs 整合法对比")
print("═" * 70)

# 混合法：等权持有 6 个单因子指数
# 使用之前模拟的各策略收益
mix_returns = []
for t in range(N_MONTHS):
    mix_ret = np.mean([
        strategy_returns['Equal_Weight'][t],
        strategy_returns['Minimum_Variance'][t],
        strategy_returns['Risk_Parity'][t],
        strategy_returns['Fundamental_BM'][t],
        strategy_returns['Dividend_Weight'][t],
        strategy_returns['Market_Cap_Weighted'][t],  # 基准
    ])
    mix_returns.append(mix_ret)

# 整合法：综合考虑多因子选股（简化：选择综合因子得分最高的股票）
integrate_returns = []
for t in range(N_MONTHS):
    # 模拟综合因子得分
    np.random.seed(42 + t)
    market_cap = np.random.lognormal(mean=10, sigma=1, size=N_STOCKS)
    log_cap = np.log(market_cap)
    btm = 1.5 - 0.08 * (log_cap - log_cap.mean()) + np.random.normal(0, 0.5, N_STOCKS)
    btm = np.clip(btm, 0.1, 3.0)

    # 综合因子得分
    score = (btm - btm.mean()) / btm.std() - (log_cap - log_cap.mean()) / log_cap.std()

    # 选择得分最高的 20% 股票
    threshold = np.percentile(score, 80)
    selected_mask = score >= threshold

    # 收益
    ret = (1.0 - 0.2 * (log_cap - log_cap.mean()) + 1.5 * (btm - btm.mean())
           + np.random.normal(0, 5, N_STOCKS))

    # 市值加权
    w = market_cap[selected_mask] / market_cap[selected_mask].sum()
    integrate_returns.append(w @ ret[selected_mask])

# 计算指标
for name, rets in [('Mix (等权持有6个因子)', mix_returns), ('Integrate (多因子选股)', integrate_returns)]:
    rets_arr = np.array(rets)
    ann_ret = np.mean(rets_arr) * 12
    ann_vol = np.std(rets_arr, ddof=1) * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    print(f"\n  {name}:")
    print(f"    年化收益率: {ann_ret:.2f}%")
    print(f"    年化波动率: {ann_vol:.2f}%")
    print(f"    Sharpe Ratio: {sharpe:.3f}")

print(f"\n💡 书中的结论：")
print(f"   整合法的 Sharpe Ratio 通常高于混合法")
print(f"   但整合法的持仓集中度和换手率更高，可投资性打折扣")
print(f"   哪种方法更好取决于投资目标和应用场景")

---

## 9. 因子配置权重

### 📖 书中的讨论

> 配置因子也可以用多种加权方式：
> - **简单多样化**：等权重
> - **波动率倒数**：低波动因子获得高权重
> - **风险平价**：各因子风险贡献相等
> - **最大分散度**：最大化分散化收益
> - **因子动量加权**：过去表现好的因子获得高权重

### 📐 配置方法公式

| 方法 | 公式 |
|------|------|
| 简单多样化 | $w_k = 1/K$ |
| 波动率倒数 | $w_k = (1/\sigma_k) / \sum(1/\sigma_j)$ |
| 风险平价 | $w_k \sigma_k = w_j \sigma_j$ |
| 最大分散度 | $\max_w \frac{w' \sigma}{\sqrt{w' \Sigma w}}$ |
| 因子动量 | $w_k \propto \bar{r}_k / \sigma_k$ |

In [ ]:
# ========== 因子配置权重对比 ==========
# 使用各策略的收益时间序列作为"因子"
factor_names = ['Equal_Weight', 'Minimum_Variance', 'Risk_Parity', 'Fundamental_BM', 'Dividend_Weight']
factor_rets = np.array([strategy_returns[name] for name in factor_names])  # K × T

# 计算各因子的统计量
factor_mean = factor_rets.mean(axis=1)  # K
factor_vol = factor_rets.std(axis=1, ddof=1)  # K
factor_cov = np.cov(factor_rets)  # K × K

K = len(factor_names)

print("📊 各因子（策略）的统计量")
print("─" * 60)
for name, mean, vol in zip(factor_names, factor_mean, factor_vol):
    print(f"  {name}: 均值={mean:.4f}%, 波动率={vol:.4f}%, SR={mean/vol:.3f}")

# 1. 简单多样化
w_simple = np.ones(K) / K

# 2. 波动率倒数
w_inv_vol = (1 / factor_vol) / (1 / factor_vol).sum()

# 3. 风险平价（简化）
w_risk_parity = w_inv_vol  # 简化版

# 4. 最大分散度
def neg_diversification_ratio(w, cov):
    port_vol = np.sqrt(w @ cov @ w)
    weighted_vol = w @ np.sqrt(np.diag(cov))
    return -weighted_vol / port_vol

constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}
bounds = [(0, 1) for _ in range(K)]
w0 = np.ones(K) / K
result = minimize(neg_diversification_ratio, w0, args=(factor_cov,),
                  method='SLSQP', bounds=bounds, constraints=constraints)
w_max_div = result.x

# 5. 因子动量（Sharpe Ratio 加权）
factor_sr = factor_mean / factor_vol
w_momentum = factor_sr / factor_sr.sum()

# 对比
print(f"\n📊 各配置方法的权重")
print("═" * 80)
weight_df = pd.DataFrame({
    'Factor': factor_names,
    'Simple': w_simple,
    'Inv_Vol': w_inv_vol,
    'Risk_Parity': w_risk_parity,
    'Max_Div': w_max_div,
    'Momentum': w_momentum
})
print(weight_df.to_string(index=False))

print(f"\n💡 书中的结论：")
print(f"   各种配置方法的差别不大")
print(f"   简单多样化（等权重）已经非常优秀")
print(f"   复杂方法需要更多参数，估计误差可能抵消理论优势")

---

## 10. 核心概念回顾

### 📌 Smart Beta
- **定义**: 在某类资产内，以更高收益或更低风险为目标，针对一个或多个因子确定持仓和权重
- **定位**: 介于被动投资和主动投资之间
- **优势**: 规则透明、成本低、因子暴露明确

### 📌 常见 Smart Beta 策略
- **等权重**: $w_i = 1/N$，隐含偏向小市值
- **最小方差**: $\min w'\Sigma w$，偏向低波动股票
- **风险平价**: 各股票风险贡献相等
- **基本面加权**: 按 B/M 等基本面指标加权，偏向价值股
- **高股息**: 按股息率加权，偏向高分红股票

### 📌 因子指数化方法
- **纯因子指数**: 数学构建，不可投资
- **多空因子指数**: 资金中性，可投资性低
- **高暴露因子指数**: 平衡因子暴露和可投资性
- **高容量因子指数**: 包含所有股票，按因子和市值加权
- **市场指数**: 市值加权，可投资性最高

### 📌 混合法 vs 整合法
- **混合法**: 持有多个单因子基金，简单直观
- **整合法**: 持有一个多因子基金，Sharpe Ratio 通常更高
- **选择**: 取决于投资目标和应用场景

### 📌 因子配置
- **简单多样化**: 等权重，已非常优秀
- **波动率倒数**: 低波动因子高权重
- **风险平价**: 各因子风险贡献相等
- **最大分散度**: 最大化分散化收益
- **因子动量**: 过去表现好的因子高权重

### 🔗 完整流程
```
选择因子 → 编制指数 → 确定权重 → 构建组合
    ↓           ↓           ↓           ↓
 价值/质量    规则透明    等权/优化    纯多头
 低波/动量    可复制      市值调整    高可投资性
```

### 📝 策略对比汇总

| 策略 | 核心思想 | 因子偏向 | 优点 | 缺点 |
|------|---------|---------|------|------|
| 等权重 | 每只股票权重相同 | 小市值 | 简单透明 | 换手率高 |
| 最小方差 | 最小化组合波动 | 低波动 | 降低风险 | 参数估计 |
| 风险平价 | 风险贡献相等 | 低波动 | 风险分散 | 参数估计 |
| 基本面加权 | 按 B/M 加权 | 价值 | 逻辑清晰 | 因子选择 |
| 高股息 | 按股息率加权 | 红利 | 现金流稳定 | 行业集中 |

### ❌ 常见误区

#### 误区 1: Smart Beta 一定能战胜市场
**✓ 正确理解**: Smart Beta 的因子溢价具有周期性，在某些时期可能跑输市场。投资者需要理解因子的周期性并坚持长期持有。

#### 误区 2: 越复杂的配置方法越好
**✓ 正确做法**: 书中的实证表明，简单多样化（等权重）已经非常优秀，复杂方法需要更多参数，估计误差可能抵消理论优势。

#### 误区 3: 只看收益率不看因子暴露
**✓ 正确做法**: Smart Beta 的核心是因子暴露。如果一个基金收益率高但因子暴露不明确，可能是运气而非因子溢价。

#### 误区 4: 忽视因子拥挤度
**✓ 正确做法**: 当某类因子受到追捧时，会造成因子过度拥挤。在"踩踏"发生前需提前预警，关注因子拥挤度指标。

#### 误区 5: 把 Smart Beta 当作短期交易工具
**✓ 正确做法**: Smart Beta 设计初衷是长期投资。书中的实证显示，持有因子指数超过 7 年时胜率高达 90% 以上。频繁交易会侵蚀因子溢价。